# Effective dimension and correlation-spectrum purity across QNN layouts

Loads the pickled results produced by `effdim_basismask_QNN.py` for a set of QNN measurement-layer layouts and reproduces the paper's comparison plots: basis-set size |B_X|/D and |B_Theta~|/K, correlation-spectrum purity tr(S^4), FIM purity tr(F_hat^2), and normalized effective dimension d_eff_hat, each shown as a function of QNN measurement layout.

In [ ]:
# Importing necessary packages
import sys
import os
import importlib
import re
import pickle

import numpy as np

import matplotlib.pyplot as plt


# Current path for importing custom functions
path_base = '/home/b/b309245/FIM_Training_Bias_RegressionModels/fourier_models_training_and_fim/'
sys.path.insert(0, path_base + 'useful_functions')

import model_constructor_functions
importlib.reload(model_constructor_functions)

import ortho_matrices_functions
importlib.reload(ortho_matrices_functions)

import tensor_network_functions_np
importlib.reload(tensor_network_functions_np)

import FIM_functions_jax
importlib.reload(FIM_functions_jax)

import compute_basis_functions_masks_qnns
importlib.reload(compute_basis_functions_masks_qnns)
import compute_basis_functions_masks_qnns as basis_masks

Re-declare the same QNN-architecture and experiment parameters (`no_qubits`, `qnn_layout`, `meas_layer_layout_vec`, `no_samples`, `no_matrix_realiz`) used in `effdim_basismask_QNN.py`, needed both to reconstruct the basis-function masks below and to match the pickled result filenames (`name_end_0`).

In [ ]:
results_folder = '/work/bd1179/b309245/fourier_models_train_and_FIM/effdim_basismask_QNN/'

# No. of qubits
no_qubits = 4
name_no_qubits = str(no_qubits)

# Layouts of QNNs
qnn_layout = ['inp', 'var', 'inp', 'var']
ent_layer_layout = 0
meas_layer_layout_vec = [0,
                         [(0, 1)],
                         [(0, 1), (2, 3)],
                         [(0, 1), (1, 2)], 
                         [(0, 1), (1, 2), (2, 3)],
                         1,
                         2]
name_meas_layer_layout_vec = ['NONE',
                              '1PAIR',
                              '2PAIR_SEP',
                              '2PAIR_NNS',
                              '3PAIR_NNS',
                              '1PBC',
                              '2PBC']


# No. of random parameter samples for evaluating normalized eff. dim.
no_samples = 200
no_par_samples_name = str(no_samples)

# No. of random orthogonal matrix realizations per S decay exponent
no_matrix_realiz = 50
no_V_samples_name = str(no_matrix_realiz)

# No. of layers, parameters, features
no_var_layers = 0
no_enc_layers = 0
for ll in qnn_layout:
    if (ll=='var'):
        no_var_layers = no_var_layers + 1
    if (ll=='inp'):
        no_enc_layers = no_enc_layers + 1
no_params = no_qubits * no_var_layers
no_encodings = no_qubits * no_enc_layers
max_freq = no_enc_layers
no_features = no_qubits
name_no_var_layers = str(no_var_layers)
name_no_enc_layers = str(no_enc_layers)
name_no_params = str(no_params)

# Local and global dimensions
d_loc_ins = 2 * max_freq + 1
d_loc_par = 3
name_d_loc_ins = str(d_loc_ins)
name_d_loc_par = str(d_loc_par)
D_ins = d_loc_ins ** no_features
D_pars = d_loc_par ** no_params
D_tot = D_ins * D_pars

name_end_0 = ('_NoQubits' + name_no_qubits + '_NoParams' + name_no_params + '_NoVarLayers' + name_no_var_layers + 
              '_NoEncLayers' + name_no_enc_layers + '_NsamplesPar' + no_par_samples_name + '_NsamplesV' + no_V_samples_name + 
              '__MeasLayout')

Recompute, for each measurement layout, the backward light-cones and the basis-function masks (over the input basis {e_mu(x)} and the parameter basis {iota_nu(theta)} separately) via `compute_basis_functions_masks_qnns`.

In [ ]:
dict_bases_layouts = dict()
for i in range(len(name_meas_layer_layout_vec)):
    name_layout = name_meas_layer_layout_vec[i]
    meas_layer_layout = meas_layer_layout_vec[i]
    BWLC_dict, params_dict, BWLC_ins_dict, inputs_dict = basis_masks.qnn_qubits_backward_lightcones(no_qubits, qnn_layout, 
                                                                                                    ent_layer_layout, meas_layer_layout)
    basis_mask = basis_masks.basis_functions_mask_vector_qnn(no_qubits, BWLC_dict, BWLC_ins_dict, no_params, 
                                                             d_loc_par, no_features, d_loc_ins)
    basis_inputs_mask, basis_params_mask = basis_masks.inputs_params_basis_mask_vectors_qnn(no_qubits, BWLC_dict, BWLC_ins_dict, 
                                                                                            no_params, d_loc_par, no_features, d_loc_ins)
    dict_layout = dict()
    dict_layout['BWLC_dict'] = BWLC_dict
    dict_layout['params_dict'] = params_dict
    dict_layout['BWLC_ins_dict'] = BWLC_ins_dict
    dict_layout['inputs_dict'] = inputs_dict
    dict_layout['basis_mask'] = basis_mask
    dict_layout['basis_inputs_mask'] = basis_inputs_mask
    dict_layout['basis_params_mask'] = basis_params_mask
    dict_bases_layouts[name_layout] = dict_layout

Load, for each layout whose results file is found in `results_folder`, the pickled dict of basis-set size, correlation-spectrum purity tr(S^4), average FIM purity, and normalized effective dimension produced by `effdim_basismask_QNN.py`.

In [ ]:
namefileini = 'norm_eff_dim' + name_end_0
listallfiles = [f for f in os.listdir(results_folder) if (f.startswith(namefileini))]
print(len(listallfiles))

layouts_found = []
for filename in listallfiles:
    res = re.findall((namefileini + '(\S+).pkl'), filename)
    layout = res[0]
    layouts_found.append(layout)


dict_exps = dict()
for i in range(len(name_meas_layer_layout_vec)):
    name_layout = name_meas_layer_layout_vec[i]
    if name_layout in layouts_found:
        filename = namefileini + name_layout + '.pkl'
        path_file = os.path.join(results_folder, filename)
        with open(path_file, 'rb') as f:
            dict_norm_ed = pickle.load(f)
        dict_exps[name_layout] = dict_norm_ed

Plot the relative input basis-set size |B_X|/D across QNN measurement layouts.

In [ ]:
y_lbl = r'$|\mathcal{B}_{\mathcal{X}}|/D$'

fs = 28
figsize = (5.5,2.5)
plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

clrs = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:olive']

cc = 1
for i in range(len(name_meas_layer_layout_vec)):
    name_layout = name_meas_layer_layout_vec[i]
    if name_layout in layouts_found:
        dict_layout = dict_bases_layouts[name_layout]
        basis_inputs_mask = dict_layout['basis_inputs_mask']
        basis_params_mask = dict_layout['basis_params_mask']
        y = np.sum(basis_inputs_mask) / D_ins
        x = cc
        ax.plot(x, y, '.', color=clrs[i], markersize=17)
        cc = cc + 1

#ax.legend(fontsize=22, loc='lower left')
#ax.legend(fontsize=22)
ax.set_xlabel(r'$\mathrm{layout}$', fontsize=fs)
ax.set_ylabel(y_lbl, fontsize=fs)
#ax.set_xscale('log')
ax.set_xticks([])
ax.set_yscale('log')
#ax.set_ylim([0.725, 0.86])
fig.savefig('InputBasisSize_layouts.png', bbox_inches='tight')
fig.savefig('InputBasisSize_layouts.pdf', bbox_inches='tight')

Plot the relative parameter basis-set size |B_Theta_tilde|/K across QNN measurement layouts.

In [ ]:
y_lbl = r'$|\tilde{\mathcal{B}}_{\Theta}|/K$'
#to_plot = 'S_purity_all'; y_lbl = r'$\mathrm{tr}\,S^4$'
#to_plot = 'avg_FIM_purity_all'; y_lbl = r'$\mathrm{tr}\,\hat{F}^2$'
#to_plot = 'norm_eff_dim_all'; y_lbl = r'$\hat{d}_{\mathrm{eff}}$'

fs = 28
figsize = (5.5,2.5)
plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

clrs = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:olive']

cc = 1
for i in range(len(name_meas_layer_layout_vec)):
    name_layout = name_meas_layer_layout_vec[i]
    if name_layout in layouts_found:
        dict_layout = dict_bases_layouts[name_layout]
        basis_inputs_mask = dict_layout['basis_inputs_mask']
        basis_params_mask = dict_layout['basis_params_mask']
        y = np.sum(basis_params_mask) / D_pars
        x = cc
        ax.plot(x, y, '.', color=clrs[i], markersize=17)
        cc = cc + 1

#ax.legend(fontsize=22, loc='lower left')
#ax.legend(fontsize=22)
ax.set_xlabel(r'$\mathrm{layout}$', fontsize=fs)
ax.set_ylabel(y_lbl, fontsize=fs)
#ax.set_xscale('log')
ax.set_xticks([])
ax.set_yscale('log')
#ax.set_ylim([0.725, 0.86])
fig.savefig('ParamsBasisSize_layouts.png', bbox_inches='tight')
fig.savefig('ParamsBasisSize_layouts.pdf', bbox_inches='tight')

Plot the correlation-spectrum purity tr(S^4) across QNN measurement layouts (random Gamma realizations per layout), together with the flat-spectrum lower bound 1/|B_X|.

In [ ]:
#to_plot = 'nnz_ratio_all'; y_lbl = r'$\mathrm{NNZ}/(DK)$'
to_plot = 'S_purity_all'; y_lbl = r'$\mathrm{tr}\,S^4$'
#to_plot = 'avg_FIM_purity_all'; y_lbl = r'$\mathrm{tr}\,\hat{F}^2$'
#to_plot = 'norm_eff_dim_all'; y_lbl = r'$\hat{d}_{\mathrm{eff}}$'

fs = 28
figsize = (6,3)
plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

clrs = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:olive']

y_lb = []
for i in range(len(name_meas_layer_layout_vec)):
    name_layout = name_meas_layer_layout_vec[i]
    if name_layout in layouts_found:
        dict_layout = dict_bases_layouts[name_layout]
        basis_inputs_mask = dict_layout['basis_inputs_mask']
        basis_params_mask = dict_layout['basis_params_mask']
        y = 1.0 / np.sum(basis_inputs_mask)
        y_lb.append(y)
y_lb = np.asarray(y_lb)

cc = 1
for i in range(len(name_meas_layer_layout_vec)):
    name_layout = name_meas_layer_layout_vec[i]
    if name_layout in layouts_found:
        dict_norm_ed = dict_exps[name_layout]
        dict_layout = dict_bases_layouts[name_layout]
        basis_inputs_mask = dict_layout['basis_inputs_mask']
        y = dict_norm_ed[to_plot]
        x = cc * np.ones(len(y))
        ax.plot(x, y, '.', color=clrs[i], markersize=17)
        cc = cc + 1
x = np.arange(0, len(y_lb)) + 1
ax.plot(x, y_lb, '--', color='k', linewidth=3, label=r'$\mathrm{lower\;bound:\;}|\mathcal{B}_{\mathcal{X}}|^{-1}$')

#ax.legend(fontsize=22, loc='lower left')
ax.legend(fontsize=26)
ax.set_xlabel(r'$\mathrm{layout}$', fontsize=fs)
ax.set_ylabel(y_lbl, fontsize=fs)
#ax.set_xscale('log')
ax.set_xticks([])
ax.set_yscale('log')
#ax.set_ylim([0.725, 0.86])
fig.savefig('trS4_layouts.png', bbox_inches='tight')
fig.savefig('trS4_layouts.pdf', bbox_inches='tight')

Plot the rescaled purity |B_X| * tr(S^4) across layouts, isolating deviations from the flat-spectrum lower bound of 1.

In [ ]:
#to_plot = 'nnz_ratio_all'; y_lbl = r'$\mathrm{NNZ}/(DK)$'
to_plot = 'S_purity_all'; y_lbl = r'$|\mathcal{B}_{\mathcal{X}}|\,\mathrm{tr}\,S^4$'
#to_plot = 'avg_FIM_purity_all'; y_lbl = r'$\mathrm{tr}\,\hat{F}^2$'
#to_plot = 'norm_eff_dim_all'; y_lbl = r'$\hat{d}_{\mathrm{eff}}$'

fs = 28
figsize = (6,3)
plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

clrs = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:olive']

y_lb = []
for i in range(len(name_meas_layer_layout_vec)):
    name_layout = name_meas_layer_layout_vec[i]
    if name_layout in layouts_found:
        dict_layout = dict_bases_layouts[name_layout]
        basis_inputs_mask = dict_layout['basis_inputs_mask']
        basis_params_mask = dict_layout['basis_params_mask']
        y = 1.0 / np.sum(basis_inputs_mask)
        y_lb.append(y)
y_lb = np.asarray(y_lb)

cc = 1
for i in range(len(name_meas_layer_layout_vec)):
    name_layout = name_meas_layer_layout_vec[i]
    if name_layout in layouts_found:
        dict_norm_ed = dict_exps[name_layout]
        dict_layout = dict_bases_layouts[name_layout]
        basis_inputs_mask = dict_layout['basis_inputs_mask']
        y = dict_norm_ed[to_plot] / y_lb[i]
        x = cc * np.ones(len(y))
        ax.plot(x, y, '.', color=clrs[i], markersize=17)
        cc = cc + 1

#ax.legend(fontsize=22, loc='lower left')
#ax.legend(fontsize=26)
ax.set_xlabel(r'$\mathrm{layout}$', fontsize=fs)
ax.set_ylabel(y_lbl, fontsize=fs)
#ax.set_xscale('log')
ax.set_xticks([])
#ax.set_yscale('log')
#ax.set_ylim([0.725, 0.86])
fig.savefig('trS4_times_NoInBasis_layouts.png', bbox_inches='tight')
fig.savefig('trS4_times_NoInBasis_layouts.pdf', bbox_inches='tight')

Plot the normalized effective dimension d_eff_hat (Eq. 12) across QNN measurement layouts.

In [ ]:
#to_plot = 'nnz_ratio_all'; y_lbl = r'$\mathrm{NNZ}/(DK)$'
#to_plot = 'S_purity_all'; y_lbl = r'$\mathrm{tr}\,S^4$'
#to_plot = 'avg_FIM_purity_all'; y_lbl = r'$\mathrm{tr}\,\hat{F}^2$'
to_plot = 'norm_eff_dim_all'; y_lbl = r'$\hat{d}_{\mathrm{eff}}$'

fs = 28
figsize = (6,3)
plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

clrs = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:olive']

cc = 1
for i in range(len(name_meas_layer_layout_vec)):
    name_layout = name_meas_layer_layout_vec[i]
    if name_layout in layouts_found:
        dict_norm_ed = dict_exps[name_layout]
        y = dict_norm_ed[to_plot]
        x = cc * np.ones(len(y))
        ax.plot(x, y, '.', color=clrs[i], markersize=17)
        cc = cc + 1

#ax.legend(fontsize=22, loc='lower left')
#ax.legend(fontsize=22)
ax.set_xticks([])
ax.set_xlabel(r'$\mathrm{layout}$', fontsize=fs)
ax.set_ylabel(y_lbl, fontsize=fs)
#ax.set_xscale('log')
#ax.set_yscale('log')
#ax.set_ylim([0.725, 0.86])
fig.savefig('EffDim_layouts.png', bbox_inches='tight')
fig.savefig('EffDim_layouts.pdf', bbox_inches='tight')

Plot the average (normalized) FIM purity tr(F_hat^2) across QNN measurement layouts.

In [ ]:
#to_plot = 'nnz_ratio_all'; y_lbl = r'$\mathrm{NNZ}/(DK)$'
#to_plot = 'S_purity_all'; y_lbl = r'$\mathrm{tr}\,S^4$'
#to_plot = 'avg_FIM_purity_all'; y_lbl = r'$\mathrm{tr}\,\hat{F}^2$'
to_plot = 'avg_FIM_purity_all'; y_lbl = r'$\overline{\mathrm{tr}(\hat{F}^2)}$'

fs = 28
figsize = (6,3)
plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

clrs = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:olive']

cc = 1
for i in range(len(name_meas_layer_layout_vec)):
    name_layout = name_meas_layer_layout_vec[i]
    if name_layout in layouts_found:
        dict_norm_ed = dict_exps[name_layout]
        y = dict_norm_ed[to_plot]
        x = cc * np.ones(len(y))
        ax.plot(x, y, '.', color=clrs[i], markersize=17)
        cc = cc + 1

#ax.legend(fontsize=22, loc='lower left')
#ax.legend(fontsize=22)
ax.set_xticks([])
ax.set_xlabel(r'$\mathrm{layout}$', fontsize=fs)
ax.set_ylabel(y_lbl, fontsize=fs)
#ax.set_xscale('log')
#ax.set_yscale('log')
#ax.set_ylim([0.725, 0.86])
fig.savefig('FIMpurity_layouts.png', bbox_inches='tight')
fig.savefig('FIMpurity_layouts.pdf', bbox_inches='tight')